# Example 4.1  
4X4 grid world page 76  

**Bellman Expectation Equation**

The full equation for the state-value function is:

$
v_\pi(s) = \sum_a \pi(a \mid s) \sum_{s',r} p(s', r \mid s, a)\left[ r + \gamma v(s') \right]
$

---

**Problem-Specific Assumptions (Gridworld)**

- Uniform random policy:  $
\pi(a \mid s) = 0.25 \quad \text{for all } a \in \{\text{up, right, down, left}\}
$

- Deterministic transitions:  $
p(s', r \mid s, a) = 1
$

- Discount factor:  $
\gamma = 1
$

**Simplified Equation**

Because transitions are deterministic, the inner sum collapses:

$
v_\pi(s) = \sum_a \pi(a \mid s)\left[ r + v(s') \right]
$

Substituting the uniform policy:

$
v_\pi(s) = \frac{1}{4} \sum_a \left[ r + v(s') \right]
$

Note: $p_\pi(s)$ is simple expectation.  We can also use the optimal approach.  Also greedy.  
In that case, we chose $\pi'(s) = \max_aq_\pi(s,a)$  

I implement both here.  Interesting to note that the convergence time using expectation is in the 271 vs. 2 for optimal policy.

---

**Final Form for This Problem**

Since $r = -1$ at every step:

$
v(s) = -1 + \frac{1}{4} \sum_a v(s')
$

---

**Action-Value Function (for reference)**

$
q(s,a) = r + v(s')
$

**The Determistics**

Can move left, right, up, down  

1. If move left the following can happen:  
Use value from left cell to calculate  
   * If on left edge (4, 8, 12) then take -1  
   * If 1 take 0  
   * Otherwiser take value of left cell.    

2. If Move right the following can happen:  
Use value from rigth cell to calculate Belman update 
     * If on right edge (3, 7, 11) then take -1 
     * if 14 take 0
     * otherwise take value from right cell.
3. If Move up
   * If on top Edge (1, 2, 3) take -1 
   * If 4, take 0 
   * Otherwise take value from cell s - 4 
4. If Move down 
   * If on bottom Edge (13, 13, 14) take -1 
   * If 11, take 0 
   * otherwise take value from cell s + 4 


In [1]:
# Initialize all values to 0
V = [0] * 16

# not used for initial case
policy = dict(left=.25, right = .25, up = .25, down = .25)

In [76]:
def getValInDir(s: int, direction:str, V:float) -> float:
    '''
    Gets the value in the given direction from current state s.

    int is cell number.  Will be 0-15
    direction in {up, right, down, left}

    '''
    v_s_next = 0
    if s in (0, 15):
        return(0)
    
    if direction == 'up':
        if s in (1,2,3):
            v_s_next = V[s] 
        else:
            v_s_next = V[s-4]

    elif direction == 'right':
        if s in (3, 7, 11):
            v_s_next = V[s] 
        else:
            v_s_next = V[s+1]

    elif direction == 'down':
        if s in (12, 13, 14):
            v_s_next = V[s] 
        else:
            v_s_next = V[s+4]

    elif direction == 'left':
        if s in (4, 8, 12):
            v_s_next = V[s] 
        else:
            v_s_next = V[s-1]

    return(v_s_next)

In [77]:
def print_V(V=V):
    for i in range(len(V)):
        v_i = round(V[i], 1)
        if (i+1)%4 == 0 and i > 0:
            print(v_i, end = "\n")
        else:
            print(v_i, end = " ")

In [202]:
def getUpdate(s, V=V, update_method = "expected"):
    v_up = getValInDir(s, 'up', V)
    v_right = getValInDir(s, 'right', V)
    v_down = getValInDir(s, 'down', V)
    v_left = getValInDir(s, 'left', V) 

    if update_method == "expected":
        updateValue = -1 + 1/4*(v_up + v_right + v_down+ v_left)
    elif update_method == "optimality":
        updateValue = max(
            -1 + v_up, -1+v_right, -1 + v_down, -1 + v_left 
        )

    return(updateValue)

In [204]:
V = [0] + [-1] * 14 + [0]
getUpdate(1, V, update_method='optimality')

-1

In [80]:
V_test = list(range(16))
V = [0] + [-1] * 14 + [0]

In [207]:
def runSim(gamma = 1e-10, maxIterations = 1000, update_method='expected'):
    V = [0] + [-1] * 14 + [0]
    i = 0
    while i <= maxIterations:
        maxDelta = -1
        for s in range(1, 15):
            next_value = getUpdate(s, V, update_method=update_method) 
            maxDelta = max(maxDelta, abs(next_value - V[s]))
            V[s] = next_value

        if maxDelta < gamma:
            break
        i+= 1
    if i > maxIterations:
        print(f"Did not converge at {maxIterations} iterations. MaxDelta: {maxDelta}")
    else:
        print(f"Converged in {i} iterations")

    return(V)


In [210]:

V_Max = runSim(gamma=1e-10, maxIterations=1000, update_method='optimality')
print_V(V_Max)

Converged in 2 iterations
0 -1 -2 -3
-1 -2 -3 -2
-2 -3 -2 -1
-3 -2 -1 0


In [212]:
V_Max = runSim(gamma=1e-10, maxIterations=1000, update_method='expected')
print_V(V_Max)

Converged in 271 iterations
0 -14.0 -20.0 -22.0
-14.0 -18.0 -20.0 -20.0
-20.0 -20.0 -18.0 -14.0
-22.0 -20.0 -14.0 0
